# Model Training & Evaluation
该 notebook 仅负责模型训练与评估。
请先运行 `data_explorer.ipynb` 的 Step 1-4 生成 `train/val/test_features.csv`。

In [5]:
# Step 5: 调用独立脚本进行高强度模型搜索与稳健训练（验证优先）
import importlib
import eeg_model_train_eval as trainer

importlib.reload(trainer)

result = trainer.run_training(
    data_dir='.',
    model_path='best_model.joblib',
    encoder_path='label_encoder.pkl',
    report_path='eeg_1dcnn_report.json',
    epochs=180,
    batch_size=16,
    learning_rate=8e-4,
    patience=28,
    robust_seed_offsets=(0, 7, 19),
    use_smote=True,
    use_calibration=True,
    cv_folds=4,
    cv_weight=0.20,
    selection_mode='val_priority',
    calibration_min_gain=0.001,
 )

print(f"最优模型: {result['best_model']}")
print(f"最优验证准确率: {result['best_val_accuracy']:.4f} ± {result['best_val_accuracy_std']:.4f}")
print(f"最优鲁棒分数: {result['best_robust_score']:.4f}")
print(f"测试集准确率: {result['test_accuracy']:.4f}")

鲁棒评估随机种子: [42, 49, 61]
SMOTE已启用: 92 -> 96 样本

===== 训练候选模型：EEG 深度网络 =====
[EEG-CNN] Epoch 001 | train_acc=0.2500 | val_acc=0.2692
[EEG-CNN] Epoch 002 | train_acc=0.2717 | val_acc=0.2692
[EEG-CNN] Epoch 003 | train_acc=0.3804 | val_acc=0.3077
[EEG-CNN] Epoch 004 | train_acc=0.3913 | val_acc=0.2692
[EEG-CNN] Epoch 005 | train_acc=0.3804 | val_acc=0.2692
[EEG-CNN] Epoch 006 | train_acc=0.3696 | val_acc=0.2692
[EEG-CNN] Epoch 007 | train_acc=0.3913 | val_acc=0.2692
[EEG-CNN] Epoch 008 | train_acc=0.4130 | val_acc=0.2308
[EEG-CNN] Epoch 009 | train_acc=0.4130 | val_acc=0.2692
[EEG-CNN] Epoch 010 | train_acc=0.3913 | val_acc=0.2692
[EEG-CNN] Epoch 011 | train_acc=0.3804 | val_acc=0.2692
[EEG-CNN] Epoch 012 | train_acc=0.3913 | val_acc=0.2308
[EEG-CNN] Epoch 013 | train_acc=0.3804 | val_acc=0.3077
[EEG-CNN] Epoch 014 | train_acc=0.3913 | val_acc=0.3077
[EEG-CNN] Epoch 015 | train_acc=0.4674 | val_acc=0.3077
[EEG-CNN] Epoch 016 | train_acc=0.3696 | val_acc=0.3077
[EEG-CNN] Epoch 017 | train_ac

In [6]:
# Step 6: 读取并展示评估结果与候选模型排名
import json
import numpy as np
import pandas as pd

with open('eeg_1dcnn_report.json', 'r', encoding='utf-8') as f:
    report_data = json.load(f)

print(f"最佳模型: {report_data['best_model']}")
print(f"最佳验证准确率: {report_data['best_val_accuracy']:.4f} ± {report_data['best_val_accuracy_std']:.4f}")
print(f"最佳鲁棒分数: {report_data['best_robust_score']:.4f}")
print(f"测试集准确率: {report_data['test_accuracy']:.4f}")
print('选择模式:', report_data['selection_mode'])
print('鲁棒评估种子:', report_data['robust_seeds'])
print('SMOTE:', report_data['use_smote'], '| 概率校准:', report_data['use_calibration'])
print('CV设置: folds=', report_data['cv_folds'], ', weight=', report_data['cv_weight'])
print('类别顺序:', report_data['classes'])
print('混淆矩阵:')
print(np.array(report_data['confusion_matrix']))

rank_df = pd.DataFrame(report_data['all_candidates'])
rank_df = rank_df.sort_values(['val_acc', 'val_acc_std'], ascending=[False, True]).reset_index(drop=True)
print('\n候选模型验证集排名:')
display(rank_df)

report_df = pd.DataFrame(report_data['classification_report']).T
report_df

最佳模型: svm:c5_gscale
最佳验证准确率: 0.8974 ± 0.0181
最佳鲁棒分数: 0.8742
测试集准确率: 0.7143
选择模式: val_priority
鲁棒评估种子: [42, 49, 61]
SMOTE: True | 概率校准: True
CV设置: folds= 4 , weight= 0.2
类别顺序: ['ambient', 'classical', 'jazz', 'rock']
混淆矩阵:
[[3 1 0 0]
 [0 2 1 0]
 [2 0 1 0]
 [0 0 0 4]]

候选模型验证集排名:


,name,val_acc,robust_score_mean,val_acc_std,cv_acc_mean
0,svm:c5_gscale,0.897436,0.874199,0.018131,0.781250
1,eeg_resmlp,0.846154,0.846154,0.000000,0.000000
2,lr:lr_balanced,0.846154,0.833173,0.000000,0.781250
3,ensemble_top3_weighted,0.846154,0.846154,0.024175,0.000000
4,rf:rf400_leaf2,0.807692,0.779487,0.031404,0.666667
5,et:et400,0.807692,0.798237,0.031404,0.760417
6,mlp:mlp256_128,0.705128,0.687019,0.072524,0.614583
7,eeg_cnn,0.320513,0.320513,0.036262,0.000000


,precision,recall,f1-score,support
ambient,0.600000,0.750000,0.666667,4.000000
classical,0.666667,0.666667,0.666667,3.000000
jazz,0.500000,0.333333,0.400000,3.000000
rock,1.000000,1.000000,1.000000,4.000000
accuracy,0.714286,0.714286,0.714286,0.714286
macro avg,0.691667,0.687500,0.683333,14.000000
weighted avg,0.707143,0.714286,0.704762,14.000000
